This is a ai chatbot that uses RAG --> Retrival Augmented Generation

In [1]:
pip install -U langchain langchain-community chromadb pypdf sentence-transformers transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Installing required libraries
!pip install faiss-cpu
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install transformers
!pip install langchain-community
!pip install langchain
!pip install langchain-text-splitters
!pip install -U langchain-core

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("CyBOK-chatbot.pdf")
documents = loader.load()

print(len(documents))

834


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)
docs = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(docs)} chunks.")
print(f"First chunk content:\n{docs[0].page_content}\n")
print(f"First chunk metadata:\n{docs[0].metadata}")

Split 834 documents into 4087 chunks.
First chunk content:
The Cyber Security
Body of Knowledge
Version /one.pnum./zero.pnum
/three.pnum/one.pnumst October /two.pnum/zero.pnum/one.pnum/nine.pnum
https:/ /www.cybok.org/
EDITORS
Awais Rashid University of Bristol
Howard Chivers University of York
George Danezis University College London
Emil Lupu Imperial College London
Andrew Martin University of Oxford
PROJECT MANAGER
Yvonne Rigby University of Bristol
PRODUCTION
Joseph Hallett University of Bristol

First chunk metadata:
{'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2020-04-20T09:24:36+01:00', 'author': 'Awais Rashid, Howard Chivers, George Danezis, Emil Lupu, Andrew Martin', 'title': 'The Cyber Security Body of Knowledge', 'subject': 'Cyber epistemology', 'keywords': '', 'moddate': '2020-04-20T09:24:36+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_15494/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
!pip install -U langchain-huggingface

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [9]:
#Testing
query = "What is cybersecurity governance?"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)

The Cyber Security Body Of Knowledge
www.cybok.org
KA Risk Management and Governance | October /two.pnum/zero.pnum/one.pnum/nine.pnum Page /four.pnum8
assessment methods and highlight their main uses and limitations, as well as providing point-
ers to more detailed information.
Security metrics are an ongoing topic of debate in the risk assessment and management do-
main: which system features to measure for risk, how to measure risk, and why measure risk
at all? These questions are framed in the context of existing literature on this topic. This links
into risk governance, which explains why effective governance is important to uphold cyber
security and some of the social and cultural factors that are essential to consider when devel-
oping governance frameworks. Almost all systems still include a human element of control,
which must be considered from the outset. Finally, even with well de/uniFB01ned and executed risk
assessment and management plans, it is still possible that a risk 

In [10]:
!pip install transformers accelerate

In [12]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="microsoft/phi-2",
    pad_token_id=50256
)


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [13]:
#TESTING
from transformers import pipeline

llm = pipeline("text-generation", model="microsoft/phi-2")

prompt = "Explain cybersecurity governance in simple terms."

result = llm(prompt, max_new_tokens=100)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain cybersecurity governance in simple terms.
Answer: Cybersecurity governance is a set of rules and responsibilities that keep our information and technology safe. It helps us make good decisions about how to protect ourselves from bad people who want to steal our information.

Exercise 2:
Why is it important for everyone to follow cybersecurity policies?
Answer: It is important for everyone to follow cybersecurity policies because it helps protect our information and technology from getting into the wrong hands. If everyone follows the rules, we can keep our information safe and


In [14]:
#Actual
def ask_cybok():

    question = input("Ask CyBOK: ")

    docs = vectorstore.similarity_search(question, k=3)

    context = ""
    for d in docs:
        context += d.page_content + "\n"

    prompt = f"Context:{context}\nQuestion:{question}\nAnswer:"

    result = llm(prompt, max_new_tokens=200)

    generated = result[0]["generated_text"]

    answer = generated.replace(prompt, "")

    print("\nBot:\n", answer.strip())

In [15]:
result = llm(prompt, max_new_tokens=200)
print(result[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain cybersecurity governance in simple terms.

Exercise 3:
Describe the role of cyber governance in ensuring the integrity of information systems.

Exercise 4:
Discuss the importance of policies, standards, and best practices in cybersecurity governance.

Exercise 5:
Explain why it is important for organizations to have a dedicated team for managing cybersecurity risks.

Solution to Exercise 1:
Cybersecurity governance is the process of managing cybersecurity risks and ensuring that an organization's information systems are secure. It involves the development and implementation of policies, procedures, and standards to protect sensitive information from unauthorized access, use, disclosure, disruption, modification, or destruction. Cyber governance also includes regular assessments, monitoring, and management of cybersecurity risks to mitigate potential threats and maintain the integrity and confidentiality of information systems.

Solution to Exercise 2:
Cybersecurity governance c

In [16]:
def ask_cybok():

    while True:

        question = input("\nAsk CyBOK (type 'exit' to stop): ")

        if question.lower() == "exit":
            print("Session ended.")
            break

        docs = vectorstore.similarity_search(question, k=3)

        context = ""
        for d in docs:
            context += d.page_content + "\n"

        prompt = f"""
Use the cybersecurity knowledge below to answer the question.

Context:
{context}

Question:
{question}

Answer clearly:
"""

        result = llm(prompt, max_new_tokens=200)

        generated = result[0]["generated_text"]
        answer = generated.replace(prompt, "")

        print("\nBot:", answer.strip())

In [ ]:
ask_cybok()


Ask CyBOK (type 'exit' to stop):  network


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: The network includes all the nodes and links that are connected to each other.

Solution:
In this chapter, we will learn about network security and its challenges. Network security is the protection of data and information that is transmitted over a network. A network is a set of nodes and links that are connected to each other. A node is a device that can send and receive data, such as a computer, a printer, a router, a switch, or a server. A link is a physical or virtual connection between two nodes, such as a cable, a wireless signal, or an IP address. A network can have different types of topologies, such as star, bus, ring, mesh, or tree. A network can also have different layers, such as physical, data link, network, transport, session, presentation, and application. Each layer has a specific function and protocol that defines how data is transmitted and received.

Network security is important because it prevents unauthorized access,
